# 00 — Test de Connectivité
**M1SPAR P1 Fraude** · Bloc 2 · Livrable 12h00

Vérifie : Spark · Dataset Parquet · MLflow · Redis

In [ ]:
from pyspark.sql import SparkSession
import mlflow
import redis
import os
import time
from dotenv import load_dotenv

load_dotenv()

## 1. Test Spark

In [ ]:
# Fix Java 17+ : Hadoop utilise Subject.getSubject() retire en Java 17
JAVA17_OPTS = (
    "--add-opens=java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = SparkSession.builder \
    .appName("M1SPAR-P1-ConnectTest") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.extraJavaOptions", JAVA17_OPTS) \
    .config("spark.executor.extraJavaOptions", JAVA17_OPTS) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

## 2. Test Dataset (50M transactions Parquet)

In [ ]:
DATASET_PATH = os.getenv(
    "DATASET_PATH",
    r"D:\m1spar-p1-fraude\data\fraud_dataset"
)

print(f"Chargement depuis : {DATASET_PATH}")
t0 = time.time()
df = spark.read.parquet(DATASET_PATH)
t1 = time.time()

n_lignes = df.count()
n_cols = len(df.columns)

print(f"Lignes chargees : {n_lignes:,}")
print(f"Colonnes        : {n_cols}")
print(f"Temps           : {t1-t0:.2f}s")

In [ ]:
# Aperçu schema
df.select("transaction_id", "amount", "country", "is_fraud", "transaction_date").show(3)

# Test partition pruning
t2 = time.time()
n_jour = df.filter("transaction_date = '2024-03-15'").count()
print(f"✅ Partition pruning : {n_jour:,} lignes en {time.time()-t2:.2f}s")

## 3. Test MLflow

In [ ]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("connectivity-test")

with mlflow.start_run():
    mlflow.log_param("status", "ok")
    mlflow.log_param("dataset_rows", n_lignes)
    mlflow.log_param("dataset_cols", n_cols)

print("✅ MLflow OK — http://localhost:5000")

## 4. Test Redis

In [ ]:
r = redis.Redis(
    host=os.getenv("REDIS_HOST", "localhost"),
    port=int(os.getenv("REDIS_PORT", 6379))
)
r.ping()
r.set("test_connect", "ok", ex=60)
print("✅ Redis OK")

## Bilan

In [ ]:
spark.stop()

print("=" * 55)
print("  LIVRABLE 12h00 - Tous les tests passes !")
print("=" * 55)
print(f"  Spark     : OK")
print(f"  Dataset   : {n_lignes:,} lignes / {n_cols} colonnes")
print(f"  MLflow    : http://localhost:5000")
print(f"  Redis     : localhost:6379")
print("=" * 55)